In [ ]:
# ════════════════════════════════════════════════════════════════════
#  CÀI THƯ VIỆN CHO BÀI NÀY   (CAI THU VIEN CHO BAI NAY)
# ════════════════════════════════════════════════════════════════════
# Cell này chỉ cài những gói mà RIÊNG bài này cần, và chỉ cài gói nào
# đang thiếu. Nếu máy đã đủ thư viện thì nó không thay đổi gì.
#
# Phiên bản được chọn tự động theo Python đang chạy, vì pandas,
# scikit-learn và tensorflow không có bản dựng sẵn cho mọi phiên bản:
#     Python 3.10  →  bản đã dùng để tạo ra kết quả trong notebook này
#     Python 3.12  →  bản mới hơn, kết quả chạy lại có thể lệch nhẹ
import sys
from importlib.util import find_spec

PY312 = sys.version_info >= (3, 12)

# (tên module để kiểm tra, gói cho Python 3.10, gói cho Python 3.12)
REQUIRED = [
    ("numpy",     "numpy==1.26.4",         "numpy==1.26.4"),
    ("pandas",    "pandas==1.5.1",         "pandas==2.2.3"),
    ("sklearn",   "scikit-learn==1.2.0",   "scikit-learn==1.5.2"),
    ("matplotlib", "matplotlib==3.8.2",     "matplotlib==3.8.2"),
    ("seaborn",   "seaborn==0.13.1",       "seaborn==0.13.1"),
]

thieu = [(p312 if PY312 else p310) for mod, p310, p312 in REQUIRED
         if find_spec(mod) is None]

print(f"Python {sys.version.split()[0]}  |  bài này cần {len(REQUIRED)} gói")

if thieu:
    print("Đang cài", len(thieu), "gói còn thiếu:", ", ".join(thieu))
    try:
        from IPython import get_ipython
        _ip = get_ipython()
    except Exception:
        _ip = None
    if _ip is not None:
        _ip.run_line_magic("pip", "install --quiet " + " ".join(thieu))
    else:
        import subprocess
        subprocess.run([sys.executable, "-m", "pip", "install", "--quiet",
                        *thieu], check=False)
    print("\n✅ Cài xong. Hãy KHỞI ĐỘNG LẠI KERNEL rồi chạy lại từ đầu.")
else:
    print("✅ Đã đủ thư viện, không cần cài gì thêm.\n")
    import importlib.metadata as _md
    for mod, p310, p312 in REQUIRED:
        ten = (p312 if PY312 else p310).split("==")[0].split(">=")[0]
        try:
            print(f"   {ten:<16} {_md.version(ten)}")
        except Exception:
            print(f"   {ten:<16} (không đọc được phiên bản)")


# Bài thực hành 2 — Bài tập
## Phát hiện Tấn công DDoS trong Lưu lượng Mạng bằng Học máy
**Bộ dữ liệu:** CICIDS2017 — Friday-WorkingHours-Afternoon-DDos

> Điền `???` vào các ô code bên dưới. Mỗi `???` đi kèm comment gợi ý đáp án.

In [ ]:
# Nhập thư viện
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (f1_score, precision_score, recall_score,
                              roc_auc_score, roc_curve, confusion_matrix,
                              ConfusionMatrixDisplay)
import warnings
warnings.filterwarnings('ignore')
print("Đã tải thư viện.")

## 1. Nạp và khám phá dữ liệu

In [ ]:
# ── Nạp dữ liệu ────────────────────────────────────────────────────────
df = pd.read_csv('data/network/b2_exercise.csv')
print("Kích thước:", df.shape)
print("\nPhân phối nhãn:")
print(df['Label'].value_counts())
print("\n3 dòng đầu:")
df.head(3)

In [ ]:
# ── Làm sạch: thay Inf -> NaN, sau đó fillna(0) ────────────────────────
feat_all = [c for c in df.columns if c != 'Label']
df[feat_all] = df[feat_all].replace([np.inf, -np.inf], np.nan).fillna(???)  # ??? = 0

# Mã hoá nhãn: DDoS = 1, BENIGN = 0
df['label'] = df['Label'].map({'DDoS': ???, 'BENIGN': ???})  # ??? = 1, 0
print("Số NaN sau khi làm sạch:", df[feat_all].isnull().sum().sum())
print("Nhãn đã mã hoá:", df['label'].value_counts().to_dict())

## 2. Phân tích khám phá (EDA)

In [ ]:
# ── Phân tích EDA: phân phối đặc trưng theo nhãn ───────────────────────
feat_plot = ['Flow Packets/s', 'Flow IAT Mean', 'Fwd Packet Length Mean',
             'Bwd Packet Length Mean', 'SYN Flag Count', 'ACK Flag Count']

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, feat in zip(axes.flat, feat_plot):
    for lbl, color, name in [(1, '#e74c3c', 'DDoS'), (0, '#2980b9', 'BENIGN')]:
        vals = df.loc[df['label'] == lbl, feat].clip(
            upper=np.percentile(df[feat], 99))
        ax.hist(vals, bins=40, alpha=0.55, color=color, label=name, density=True)
    ax.set_title(feat, fontsize=10)
    ax.legend(fontsize=8)
plt.suptitle('Phân phối đặc trưng: DDoS vs BENIGN', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# ── Scatter plot: Flow Packets/s vs Flow IAT Mean ───────────────────────
fig, ax = plt.subplots(figsize=(8, 5))
for lbl, color, name in [(0, '#2980b9', 'BENIGN'), (1, '#e74c3c', 'DDoS')]:
    sub = df[df['label'] == lbl].sample(
        min(1500, int((df['label'] == lbl).sum())), random_state=42)
    ax.scatter(
        sub['Flow Packets/s'].clip(upper=np.percentile(df['Flow Packets/s'], 99)),
        sub['Flow IAT Mean'].clip(upper=np.percentile(df['Flow IAT Mean'], 99)),
        alpha=0.25, s=8, color=color, label=name)
ax.set_xlabel('Flow Packets/s')
ax.set_ylabel('Flow IAT Mean (µs)')
ax.set_title('Flow Packets/s vs Flow IAT Mean — DDoS vs BENIGN')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── Thống kê mô tả theo nhãn ────────────────────────────────────────────
print("Giá trị trung vị theo nhãn:")
print(df.groupby('???')[feat_plot].median().round(2).to_string())  # ??? = 'Label'

## 3. Chọn đặc trưng — Chia tập — Chuẩn hoá

In [ ]:
# ── Chọn đặc trưng và chia tập train/val/test ───────────────────────────
feat_cols = ['Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std',
             'Fwd Packet Length Mean', 'Bwd Packet Length Mean',
             'SYN Flag Count', 'ACK Flag Count']

X = df[feat_cols].values
y = df['???'].values       # ??? = 'label'

# Chia 64% train / 16% val / 20% test (2 lần train_test_split)
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=???, random_state=42, stratify=y)   # ??? = 0.2
X_tr, X_val, y_tr, y_val = train_test_split(
    X_trainval, y_trainval, test_size=???,               # ??? = 0.2
    random_state=42, stratify=y_trainval)

# Chuẩn hoá: fit trên train, transform trên val và test (tránh data leakage)
scaler = StandardScaler()
X_tr   = scaler.fit_transform(???)                      # ??? = X_tr
X_val  = scaler.transform(X_val)
X_test = scaler.transform(X_test)

print(f"Train: {X_tr.shape}, Val: {X_val.shape}, Test: {X_test.shape}")

## 4. Huấn luyện và So sánh mô hình (Tập Validation)

In [ ]:
# ── Huấn luyện LR / RF / GBC và so sánh trên tập Validation ────────────
classifiers = {
    'LR' : LogisticRegression(max_iter=2000, class_weight='balanced',
                              C=0.1, random_state=42),
    'RF' : RandomForestClassifier(n_estimators=100, class_weight='balanced',
                                  random_state=42),
    'GBC': GradientBoostingClassifier(n_estimators=100, random_state=42),
}

results = {}
for name, model in classifiers.items():
    model.fit(???, ???)                  # ??? = X_tr, y_tr
    yp  = model.predict(???)             # ??? = X_val
    pp  = model.predict_proba(???)[:, 1] # ??? = X_val
    results[name] = {
        'F1'  : f1_score(y_val, yp, average='macro'),
        'Prec': precision_score(y_val, yp, average='macro'),
        'Rec' : recall_score(y_val, yp, average='macro'),
        'AUC' : roc_auc_score(y_val, pp),
    }

pd.DataFrame(results).T.round(4)

## 5. Độ quan trọng đặc trưng và Đường cong ROC

In [ ]:
# ── Độ quan trọng đặc trưng (Random Forest) ────────────────────────────
rf_model = classifiers['RF']
importances = rf_model.feature_importances_
feat_imp = pd.Series(importances, index=feat_cols).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 4))
feat_imp.plot.barh(ax=ax, color='#2ecc71', edgecolor='black')
ax.set_xlabel('Mức độ quan trọng')
ax.set_title('Random Forest — Độ quan trọng đặc trưng')
plt.tight_layout()
plt.show()

In [ ]:
# ── Đường cong ROC (tập Validation) ─────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 5))
colors = {'LR': '#e74c3c', 'RF': '#2980b9', 'GBC': '#27ae60'}
for name, model in classifiers.items():
    pp = model.predict_proba(X_val)[:, 1]
    fpr, tpr, _ = roc_curve(???, pp)    # ??? = y_val
    auc = roc_auc_score(y_val, pp)
    ax.plot(fpr, tpr, label=f'{name} (AUC={auc:.4f})', color=colors[name])
ax.plot([0, 1], [0, 1], 'k--')
ax.set_xlabel('Tỷ lệ dương tính giả (FPR)')
ax.set_ylabel('Tỷ lệ dương tính thật (TPR)')
ax.set_title('Đường cong ROC — Tập Validation')
ax.legend()
plt.tight_layout()
plt.show()

## 6. Đánh giá mô hình tốt nhất trên Tập Test

In [ ]:
# ── Đánh giá mô hình tốt nhất trên tập Test ────────────────────────────
best_name  = max(results, key=lambda k: results[k]['F1'])
best_model = classifiers[best_name]
print(f"Mô hình tốt nhất (theo Val F1): {best_name}")

yp_test = best_model.predict(???)           # ??? = X_test
pp_test = best_model.predict_proba(???)[:, 1]  # ??? = X_test
print(f"Test F1 : {f1_score(y_test, yp_test, average='macro'):.4f}")
print(f"Test AUC: {roc_auc_score(y_test, pp_test):.4f}")

In [ ]:
# ── Ma trận nhầm lẫn trên tập Test ──────────────────────────────────────
cm = confusion_matrix(???, yp_test)     # ??? = y_test
disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                               display_labels=['BENIGN', 'DDoS'])
disp.plot(cmap='Blues')
plt.title(f'Ma trận nhầm lẫn — {best_name} (Tập Test)')
plt.tight_layout()
plt.show()

## 7. Kết luận

In [ ]:
# ── Tổng kết ─────────────────────────────────────────────────────────────
print("=== Kết quả tổng hợp ===")
print(f"{'Mô hình':<8} {'F1':>8} {'Prec':>8} {'Rec':>8} {'AUC':>8}")
for n, r in results.items():
    print(f"{n:<8} {r['F1']:>8.4f} {r['Prec']:>8.4f} {r['Rec']:>8.4f} {r['AUC']:>8.4f}")
print()
print("Nhận xét:")
print("  LR  F1 ~0.91  — bị giới hạn bởi ranh giới tuyến tính trong không gian 7 chiều.")
print("  RF/GBC F1 ~0.997 — cây phi tuyến học được tổ hợp điều kiện phức tạp tốt hơn.")
print("  DDoS: IAT cao + gói nhỏ chiều gửi + SYN thấp → ranh giới phi tuyến so với BENIGN.")